# ADTA-DAST 5770 — Group 4 — Generative AI Q&A-Search System
**Authors:** Karan Parekh (Member 1), Sanjana Pendyala Ravinder (Member 2), Sana Mhapsekar (Member 3), Medina Maloku (Member 4)

**Project ID:** `precise-machine-471801-n5`

**Corpus bucket:** `gs://adta5770-docs-folder-group4-kp/documents/pdfs/` (100 supply-chain PDFs)

This notebook implements the full 10-phase RAG pipeline demonstrated in Dr. Nguyen's lecture on `adta-dast_5770_qasearch_ALL_PHASES_2026_LECTURES_NO_OUTPUTS.ipynb`.


## PHASE 1: SET UP DEVELOPMENT ENVIRONMENT — INSTALLATIONS — IMPORTS


### 1.1 Installations


In [ ]:
# Core GCP / Vertex AI SDKs
%pip install --upgrade google-cloud-aiplatform --quiet
%pip install --upgrade vertexai --quiet
%pip install --upgrade google-genai --quiet

In [ ]:
# LangChain + all required sub-packages
%pip install --upgrade langchain langchain-core langchain-classic langchain-community --quiet
%pip install --upgrade langchain-text-splitters --quiet
%pip install --upgrade langchain-google-community langchain-google-vertexai langchain-google-genai --quiet

In [ ]:
# PDF + OCR dependencies for unstructured document loading
!sudo apt -y -qq install tesseract-ocr libtesseract-dev
!sudo apt-get -y -qq install poppler-utils
%pip install --user --upgrade unstructured pdf2image pytesseract pdfminer.six unstructured_pytesseract --quiet
%pip install --user --upgrade pillow-heif opencv-python unstructured-inference pikepdf pypdf pi_heif --quiet

### 1.2 Authenticate with GCP and set project


In [ ]:
import sys
from google.colab import auth
from google.cloud import storage

# 1. Authenticate when running inside Google Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()

# 2. Set project ID (Group 4's GCP project)
PROJECT_ID = 'precise-machine-471801-n5'
!gcloud config set project {PROJECT_ID}

# 3. Initialize storage client
storage_client = storage.Client(project=PROJECT_ID)
print(f"Authenticated with project: {storage_client.project}")

# Verify host project
!gcloud config get-value project

### 1.3 Imports from GCP Vertex AI


In [ ]:
# ---- IMPORT FROM GCP: VERTEX AI ----
from google.cloud import aiplatform
import vertexai

# Namespace and NumericNamespace are used later for filtered vector-search retrieval
from google.cloud.aiplatform.matching_engine.matching_engine_index_endpoint import (
    Namespace,
    NumericNamespace,
)

### 1.4 Imports from LangChain (modern package structure)


In [ ]:
# NEW: chains live in langchain_classic (the package that now hosts all chain code)
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# NEW:
from langchain_google_community import GCSDirectoryLoader, GCSFileLoader

# NEW:
from langchain_core.prompts import PromptTemplate

# CHANGE: ChatPromptTemplate is imported from langchain_core.prompts because
# create_stuff_documents_chain (the replacement for RetrievalQA) requires a
# ChatPromptTemplate rather than a plain PromptTemplate.
from langchain_core.prompts import ChatPromptTemplate

# NEW (single correct import):
from langchain_text_splitters import RecursiveCharacterTextSplitter

# CHANGE: PyPDFLoader import from langchain_community.document_loaders is correct
# and remains unchanged. This is the proper location in the reorganized ecosystem.
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
# ---- IMPORT FROM GCP: VERTEX AI & LANGCHAIN API ----
# CHANGE: These imports from langchain_google_vertexai remain correct and unchanged.
# - VertexAI: LLM wrapper for Vertex AI text models (e.g., text-bison, gemini)
# - VertexAIEmbeddings: Embedding model wrapper (e.g., text-embedding-005)
# - VectorSearchVectorStore: LangChain vector store backed by Vertex AI Vector Search
# - VectorSearchVectorStoreDatastore: variant that uses Datastore for document storage
from langchain_google_vertexai import VertexAI
from langchain_google_vertexai import VertexAIEmbeddings
from langchain_google_vertexai import (
    VectorSearchVectorStore,
    VectorSearchVectorStoreDatastore,
)

In [ ]:
# ---- IMPORT OTHERS ----
import textwrap

### 1.5 Verify library versions


In [ ]:
# Check versions of the main SW platforms of the system
# GCP aiplatform & vertex ai
import vertexai
from google.cloud import aiplatform
print(f"aiplatform SDK version: {aiplatform.__version__}")
print(f"Vertex AI SDK version: {vertexai.__version__}")

# LangChain, langchain-core, langchain-community
import langchain
print(f"LangChain version: {langchain.__version__}")

from langchain_core import __version__ as langchain_core_version
print(f"langchain-core version: {langchain_core_version}")

from langchain_classic import __version__ as langchain_classic_version
print(f"langchain-classic version: {langchain_classic_version}")

from langchain_community import __version__ as langchain_community_version
print(f"langchain-community version: {langchain_community_version}")

from langchain_google_community import __version__ as langchain_google_community_version
print(f"langchain-google-community version: {langchain_google_community_version}")

### 1.6 Build system development environment (paths + region)


In [ ]:
# BUILD SYSTEM DEVELOPMENT ENVIRONMENT
# Already set PROJECT_ID: PROJECT_ID = 'precise-machine-471801-n5'

REGION = "us-central1"
BUCKET_NAME = "adta5770-docs-folder-group4-kp"   # Group 4's bucket (created during HW4)
folder_prefix = "documents/pdfs/"

BUCKET_URI = f"gs://{BUCKET_NAME}/{folder_prefix}"

print(f"BUCKET_URI: {BUCKET_URI}")

In [ ]:
# ALL PDFS ARE ALREADY IN THE BUCKET - DON'T DO ANYTHING HERE
""" COMMENT ALL
!gcloud storage cp -r gs://github-repo/documents/google-research-pdfs/* {BUCKET_URI}
"""

# Verify the PDFs are visible in the bucket
!gsutil ls {BUCKET_URI} | head -5
!echo "Total PDF count:"
!gsutil ls {BUCKET_URI} | wc -l

### 1.7 Initialize GCP AI Platform for the AI SW system


In [ ]:
# --------------------------------------------------------------------
# INITIALIZE GCP AI PLATFORM FOR AI SW SYSTEM
# --------------------------------------------------------------------
# Initialize the system (the current AI software application)
# This AI software application is associated with the GCP project as declared
# This AI software application runs at the specified GCP region
# This AI software application uses the specified GCS bucket
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)
print("GCP AI Platform initialized.")

### 1.8 Define text embedding model & index constants


In [ ]:
# --------------------------------------------------------------------
# DEFINE TEXT EMBEDDING MODEL CREDENTIALS & CREATE IT
# --------------------------------------------------------------------

# The number of dimensions for text-embedding-005 is 768.
# The text-embedding-005 model is Google's latest text embedding model,
# which replaced the older gecko models. It produces 768-dimensional vectors,
# same as gecko@003.
DIMENSIONS = 768

# Index Constants
DISPLAY_NAME_ID = "adta5770_group4_index"       # Display name for the Matching Engine index
DEPLOYED_INDEX_ID = "adta5770_group4_endpoint"  # Deployed index / endpoint ID

# Create text embedding model based on GCP Vertex AI: text-embedding-005
embedding_model = VertexAIEmbeddings(
    model_name="text-embedding-005",
    project=PROJECT_ID,
)
print(f"Embedding model ready: text-embedding-005 ({DIMENSIONS} dims)")

### 1.9 Check for existing indexes & endpoints (clean slate verification)


In [ ]:
# CHECK TO FIND ALL EXISTING INDICES AND ENDPOINTS
list_indexes = aiplatform.MatchingEngineIndex.list()
print("Existing indexes:")
print(list_indexes)
print("\n\n")

list_end_points = aiplatform.MatchingEngineIndexEndpoint.list()
print("Existing endpoints:")
print(list_end_points)

## PHASE 2: PROCESS DOCUMENTS
Make the PDF corpus ready for embedding and vectorization.


### 2.1 List blobs (PDFs) in the GCS bucket


In [ ]:
# For GCS: Google Cloud Storage access and loading
# In GCS: Each document/file is a 'blob' (SQL: Blob is a collection of text)
# Knowledge base: named as 'documents_from_blob'
# Knowledge base is a storage/data structure to store the documents

from google.cloud import storage
client = storage.Client()

for blob in client.list_blobs(BUCKET_NAME, prefix=folder_prefix):
    print(str(blob))

### 2.2 Load PDF files in the GCS bucket into a KNOWLEDGE BASE
NOTE: It can take 5 minutes or more.


In [ ]:
# --- Initialization ---
print(f"Processing documents from gs://{BUCKET_URI}")

bucket = storage_client.bucket(BUCKET_NAME)

# --- Load Documents ---
all_documents = []
blobs = bucket.list_blobs(prefix=folder_prefix)  # List blobs matching the prefix

for blob in blobs:
    # Skip directories/folders if represented as blobs, and ensure it's a PDF
    if blob.name.endswith("/") or not blob.name.lower().endswith(".pdf"):
        continue

    print(f"  Loading document: {blob.name}")

    # Use GCSFileLoader for each PDF blob
    loader = GCSFileLoader(
        project_name=PROJECT_ID,
        bucket=BUCKET_NAME,
        blob=blob.name,
    )

    # Load documents (GCSFileLoader often returns one Document per page)
    # VIP NOTES: Loader.load() only load one file at a time because it is 'GCSFileLoader'
    documents_from_blob = loader.load()

    # =========================== NEW NEW NEW =============================
    # --- Metadata Enhancement (similar to original logic) ---
    # Derive document name from the blob name
    document_name = blob.name.split("/")[-1]

    # Derive doc source prefix and suffix to match original logic
    doc_source_prefix = f"gs://{BUCKET_NAME}"
    doc_source_suffix = "/".join(blob.name.split("/")[0:-1])
    source = f"{doc_source_prefix}/{doc_source_suffix}"  # Folder path as source

    # VIP NOTES: Only one document existed in documents_from_blob (GCSFileLoader is used)
    for document in documents_from_blob:
        # Add derived metadata to each document (page)
        document.metadata["source"] = source
        document.metadata["document_name"] = document_name

    # GCSFileLoader might add other useful metadata like 'page' automatically
    # NOW: Add this document into the list all_documents
    all_documents.extend(documents_from_blob)

# The `all_documents` list now contains LangChain Document objects,
# likely one per page from all loaded PDFs.
print(f"# of document pages loaded (pre-chunking) = {len(all_documents)}")

## PHASE 3: CHUNK DOCUMENTS → CHUNKS
Split each loaded document into retrieval-sized chunks.


In [ ]:
# Chunk documents using RecursiveCharacterTextSplitter
# chunk_size ≈ 1000 tokens, chunk_overlap = 200 preserves sentence boundaries

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)

doc_splits = text_splitter.split_documents(all_documents)
print(f"# of chunks after splitting = {len(doc_splits)}")
print(f"Example chunk (first 300 chars):\n{doc_splits[0].page_content[:300]}")
print(f"Example metadata: {doc_splits[0].metadata}")

## PHASE 4: CREATE VECTOR SEARCH INDEX + DEPLOY TO ENDPOINT
**This phase takes ~30 minutes to complete** (index creation + endpoint deployment).


### 4.1 Create the GCS bucket used by the Matching Engine index


In [ ]:
# IMPORTANT CODE: DON'T MISS
# Create a GCS bucket to store the Matching Engine / Vector Search index embeddings
VSVDB_REGION = REGION
VSVDB_EMBEDDING_DIR = "adta5770-group4-vsvdb"  # Vector search DB bucket name

# Try to create the bucket; if it already exists, continue without error
!set -x && gsutil mb -p $PROJECT_ID -l us-central1 gs://$VSVDB_EMBEDDING_DIR 2>/dev/null || echo "Bucket gs://$VSVDB_EMBEDDING_DIR may already exist — continuing."
!gsutil ls gs://$VSVDB_EMBEDDING_DIR

### 4.2 Create the Matching Engine index (tree-AH, streaming updates)


In [ ]:
# Create the Matching Engine index
# STREAM_UPDATE lets us push new embeddings and query them within seconds
my_index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name=DISPLAY_NAME_ID,
    dimensions=DIMENSIONS,
    approximate_neighbors_count=150,
    leaf_node_embedding_count=500,
    leaf_nodes_to_search_percent=7,
    description="ADTA 5770 Group 4 Q&A Search Index",
    index_update_method="STREAM_UPDATE",
)
print(f"Index created. Name: {my_index.name}")
print(f"Index resource name: {my_index.resource_name}")

### 4.3 Create the Index Endpoint and deploy the index


In [ ]:
# Create the endpoint (public) then deploy the index to it.
# Deployment is the slow step — roughly 20–30 minutes.
my_index_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name=DEPLOYED_INDEX_ID,
    public_endpoint_enabled=True,
    description="ADTA 5770 Group 4 Q&A Search Endpoint",
)
print(f"Endpoint created: {my_index_endpoint.name}")

# Deploy the index to the endpoint (this is the ~30 min step)
my_index_endpoint.deploy_index(
    index=my_index,
    deployed_index_id=DEPLOYED_INDEX_ID,
)
print(f"Index deployed with deployed_index_id = {DEPLOYED_INDEX_ID}")

### 4.4 Confirm index & endpoint identities


In [ ]:
# SPECIAL CASE: ONLY ONE INDEX AND ONE ENDPOINT THAT HAS BEEN DEPLOYED
my_index = aiplatform.MatchingEngineIndex.list()[0]
my_endpoint = aiplatform.MatchingEngineIndexEndpoint.list()[0]
deployed_index_id = my_endpoint.deployed_indexes[0].id

# ====================================================================
# GET & SET index_id and end_point_id
# ====================================================================
# These variables are set for convenience in subsequent code.

# INDEX NAME and INDEX ID
index_id = my_index.name
print(f"INDEX NAME: {index_id}")

# DEPLOYED INDEX ID (DEPLOYED_INDEX_ID == deployed_index_id)
print(f"DEPLOYED INDEX ID: {deployed_index_id}")

end_point_id = my_endpoint.name
print(f"end_point_id: {end_point_id}")

## PHASE 5: CONFIGURE INDEX AS VECTOR STORE
Wrap the deployed Matching Engine index as a LangChain `VectorSearchVectorStore`.

**VIP NOTES:** Before creating the vector store using the index, the index MUST be DEPLOYED (done in Phase 4).


In [ ]:
# Embeddings API integrated with LangChain
# Create a Vector Search vector database, vsvectordb
# It is actually a GCP Vertex AI vector store
# Initialize the vector store

vsvectordb = VectorSearchVectorStore.from_components(
    project_id=PROJECT_ID,
    region=VSVDB_REGION,
    gcs_bucket_name=VSVDB_EMBEDDING_DIR,
    embedding=embedding_model,   # VertexAIEmbeddings(model_name='text-embedding-005')
    index_id=my_index.name,
    endpoint_id=my_endpoint.name,
    stream_update=True,
)
print("Vector store ready.")

## PHASE 6: ADD DOCUMENTS AS EMBEDDINGS IN MATCHING ENGINE AS INDEX
Transform chunks into 768-dim embeddings via Vertex AI Embeddings API and stream them into the index.

**NOTE:** Depending on the volume and size of documents, this step may take time.


In [ ]:
# Store docs as embeddings in Matching Engine/Vector Search index
# It may take a while since API is rate limited
# The Vertex AI embedding model API has a limit of 250 instances per prediction request.
# To avoid the '400 INVALID_ARGUMENT' error, we need to batch the texts.

texts = [doc.page_content for doc in doc_splits]
metadatas = [doc.metadata for doc in doc_splits]

# Define a batch size, slightly less than the limit (e.g., 200 or 250)
batch_size = 100  # Reduced further to ensure total token count per batch is strictly < 20000
total_batches = (len(texts) + batch_size - 1) // batch_size

for i in range(0, len(texts), batch_size):
    batch_texts = texts[i : i + batch_size]
    batch_metadatas = metadatas[i : i + batch_size]
    print(f"Adding batch {i//batch_size + 1}/{total_batches} (size: {len(batch_texts)})... ")
    vsvectordb.add_texts(texts=batch_texts, metadatas=batch_metadatas)

print("All texts added successfully.")

## PHASE 7: TEST VECTOR SEARCH INDEX TO PERFORM Q&A SEARCH
Validate that semantic search from Matching Engine is working.


In [ ]:
# Test whether search from vector store is working
results = vsvectordb.similarity_search("What are the main challenges in supply chain management?", k=2)
for i, r in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Document: {r.metadata.get('document_name', 'unknown')}")
    print(f"Content (first 300 chars): {r.page_content[:300]}")
    print()

## PHASE 8: SET UP QUERY AND RESPONSE SUBSYSTEM
Create an LLM based on an existing Google Generative AI model (Gemini 2.5 Pro).

**NOTES:** Check valid versions at https://docs.cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions


### 8.1 Instantiate the LLM


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    project=PROJECT_ID,    # Ensure project ID is passed for Vertex AI access
    location=REGION,       # Ensure region is passed for Vertex AI access
    max_output_tokens=8192,
    temperature=0.2,
    top_p=0.8,
    top_k=40,
    verbose=True,
)
print("LLM (gemini-2.5-pro) ready.")

### 8.2 Configure an instance of RetrievalQA
**VIP NOTES:**
- A Vertex AI vector search index (vector database) set up as a Vertex AI vector store can be configured as a RetrievalQA.
- RetrievalQA is a Vertex AI vector search tool specifically used for Q&A Search.
- First, need to configure a **retriever** — Retriever is used to retrieve response from the vector search index (or vector store).


In [ ]:
# Create chain to answer questions

# Number of responses that the system tries to search for
NUMBER_OF_RESULTS = 10

# The distance (similarity search) used as a threshold to search for responses
SEARCH_DISTANCE_THRESHOLD = 0.6

# Expose index to the retriever
retriever = vsvectordb.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": NUMBER_OF_RESULTS,
    },
    filters=None,
)
print("Retriever ready.")

### 8.3 Customize a prompt template


In [ ]:
"""### Customize a prompt template"""

# CHANGE (PHASE 8 FIX): The prompt template variable names must match what
# create_retrieval_chain and create_stuff_documents_chain expect:
#   - {input}   : the user's question (was {question} in the old RetrievalQA template)
#   - {context} : the retrieved document chunks (remains the same)
# Using {question} would cause a KeyError because the new chain passes the user query
# as 'input', not 'question'. The template is also wrapped in ChatPromptTemplate.from_template()
# because create_stuff_documents_chain requires a ChatPromptTemplate (not a plain PromptTemplate).
# A plain PromptTemplate causes a ValidationError.

template = """SYSTEM: You are an intelligent assistant helping the users with their questions on research papers.

Question: {input}

Strictly Use ONLY the following pieces of context to answer the question at the end. Think step-by-step and then answer.

Do not try to make up an answer:
 - If the answer to the question cannot be determined from the context alone, say \"I cannot determine the answer to that.\"
 - If the context is empty, just say \"I do not know the answer to that.\"

=============
{context}
=============

Question: {input}
Helpful Answer:"""

In [ ]:
# Create the ChatPromptTemplate from the template string
# CHANGE: create_stuff_documents_chain requires ChatPromptTemplate, not PromptTemplate.
# ChatPromptTemplate properly formats the prompt as a chat message, which is required
# by modern LangChain chain functions.
qa_prompt = ChatPromptTemplate.from_template(template)

In [ ]:
# Create the document-combining chain (replaces chain_type="stuff" in RetrievalQA)
combine_docs_chain = create_stuff_documents_chain(llm, qa_prompt)

In [ ]:
# Create the full retrieval chain (replaces RetrievalQA.from_chain_type())
retrieval_qa_chain = create_retrieval_chain(retriever, combine_docs_chain)

In [ ]:
from langchain_core.globals import set_verbose, set_debug
set_verbose(True)
# Uncomment the following line for even more detailed debug output:
# set_debug(True)

### 8.4 Utility function used to format the response


In [ ]:
"""### Utility function used to format the response"""
def formatter(result):
    print(f"Query: {result['input']}")
    print("." * 80)
    if "context" in result.keys():
        for idx, ref in enumerate(result["context"]):
            print("-" * 80)
            print(f"REFERENCE #{idx}")
            print("-" * 80)
            if "score" in ref.metadata:
                print(f"Matching Score: {ref.metadata['score']}")
            if "source" in ref.metadata:
                print(f"Document Source: {ref.metadata['source']}")
            if "document_name" in ref.metadata:
                print(f"Document Name: {ref.metadata['document_name']}")
            print("." * 80)
            print(f"Content: \n{wrap(ref.page_content)}")
    print("." * 80)
    print(f"Response: {wrap(result['answer'])}")
    print("." * 80)

In [ ]:
def wrap(s):
    return "\n".join(textwrap.wrap(s, width=120, break_long_words=False))

In [ ]:
def ask(
    query,
    k=NUMBER_OF_RESULTS,
    # search_distance=SEARCH_DISTANCE_THRESHOLD,
    filters=None,
):
    retriever.search_kwargs["k"] = k

    if filters:
        ns_filters = [Namespace(name=filters["namespace"],
                                allow_tokens=filters.get("allow_list"),
                                deny_tokens=filters.get("deny_list"))]
        retriever.search_kwargs["filters"] = ns_filters
    else:
        retriever.search_kwargs.pop("filters", None)

    result = retrieval_qa_chain.invoke({"input": query})
    formatter(result)

## PHASE 9: TEST Q&A SEARCH SYSTEM
### Q&A SEARCH with FILTERS


In [ ]:
"""### Run QA chain on sample questions\n\n### **PHASE 9:** TEST Q&A SEARCH SYSTEM"""
ask("What are the main challenges in global supply chain management?")

In [ ]:
ask("How does artificial intelligence help with inventory optimization?")

In [ ]:
ask("What are the key benefits of blockchain in supply chain management?")

In [ ]:
ask("What is supply chain resilience?")

In [ ]:
ask("How do companies manage supply chain disruptions during a pandemic?")

In [ ]:
ask("What are the main factors affecting supplier selection?")

In [ ]:
# Q&A with FILTER — restrict retrieval to a specific document
filters = {
    "namespace": "document_name",
    "allow_list": ["SustAI-SCM Intelligent Supply Chain Process Automation with Agentic AI.pdf"],
}
ask("What is agentic AI in the context of supply chain?", filters=filters)

In [ ]:
ask("What is sustainable supply chain management?")

In [ ]:
ask("What role does digital transformation play in modern supply chains?")

## PHASE 10: UNDEPLOY INDEX AND DELETE ALL INDEXES & ENDPOINTS
**Per the Design & Code Review rubric, Phase 10 is CODED but NOT executed** (to preserve the deployed index for review).

Run the cells below ONLY after the review is complete to avoid incurring hourly charges.


In [ ]:
# ============================================================
# DO NOT RUN THIS CELL UNTIL THE DESIGN & CODE REVIEW IS DONE.
# Running it will destroy the deployed index and halt further Q&A.
# ============================================================

# GENERAL CASE: MULTIPLE INDEXES AND ENDPOINTS HAVE BEEN DEPLOYED
# Uncomment and run to clean up.

# # UNDEPLOY INDEXES and DELETE ENDPOINTS
# del_index_endpoint_1 = aiplatform.MatchingEngineIndexEndpoint(my_endpoint.name)

# # Un-deploy indexes and delete end points
# del_index_endpoint_1.undeploy_all()

# # and delete end points
# del_index_endpoint_1.delete()

In [ ]:
# # DELETE INDICES

# # Get the list of indexes that have been created for the vector search system
# list_indexes = aiplatform.MatchingEngineIndex.list()
# print(f"List of indexes: {list_indexes}")

# del_index_1 = aiplatform.MatchingEngineIndex(my_index.name)
# # ... Continue to get all indexes

# del_index_1.delete()
# # ... Continue until deleting all indexes

# # Verify that all indexes have been deleted --> The list should be empty: Nothing is printed out
# list_indexes = aiplatform.MatchingEngineIndex.list()
# print(list_indexes)

# ###---------------------------- AT THIS POINT:
# # All indexes have been un-deployed and deleted.
# # All end points have been deleted

In [ ]:
# # DISPLAY LIST OF ENPOINTS and INDEXES TO CHECK

# list_indexes = aiplatform.MatchingEngineIndex.list()
# print(list_indexes)

# print("\n\n")

# list_end_points = aiplatform.MatchingEngineIndexEndpoint.list()
# print(list_end_points)

## End of notebook
**Design & Code Review checklist (per Dr. Nguyen's Announcement 55):**
- [x] High-Level System design: Completed (see HW5 Part III)
- [ ] PHASES 1, 2, 3, 4: coded + executed successfully
- [ ] PHASES 5, 6, 7, 8, 9: coded + executed successfully
- [x] PHASE 10: coded (not executed)
